# AraStudy Cloud Training (Kaggle / Colab)
Notebook جاهزة لتشغيل baseline / phase1 / phase2 / probing باستخدام `notebooks/cloud_train.py`.

In [ ]:
# 1) إعداد سريع
import os, sys, subprocess

print('Python:', sys.version)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'sentencepiece', 'pyyaml', 'numpy'], check=True)
print('Dependencies ready ✅')

In [ ]:
# 2) الحصول على المشروع
import os
import subprocess

# يمكنك override عبر متغير البيئة ARASTUDY_REPO_URL
DEFAULT_REPO_URL = 'https://github.com/faresrafat3/arastudy-morphology.git'
REPO_URL = os.environ.get('ARASTUDY_REPO_URL', DEFAULT_REPO_URL).strip()

if not os.path.exists('/kaggle/working/arastudy') and not os.path.exists('/content/arastudy'):
    base = '/kaggle/working' if os.path.exists('/kaggle/working') else '/content'
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, f'{base}/arastudy'], check=True)

PROJECT_DIR = '/kaggle/working/arastudy' if os.path.exists('/kaggle/working/arastudy') else '/content/arastudy'
os.chdir(PROJECT_DIR)
print('Project dir:', PROJECT_DIR)

In [ ]:
# 2.1) تنزيل ملفات البيانات تلقائيًا من GitHub Release (لو ناقصة)
from pathlib import Path
import urllib.request

RELEASE_BASE = 'https://github.com/faresrafat3/arastudy-morphology/releases/download/cloud-data-v1'
ASSETS = [
    ('results/tokenizers/bpe_16k.model', 'bpe_16k.model'),
    ('data/processed/train_phase1.txt', 'train_phase1.txt'),
    ('data/processed/valid.txt', 'valid.txt'),
    ('data/morphology/root_ids_lookup.npy', 'root_ids_lookup.npy'),
    ('data/morphology/token_root_map.json', 'token_root_map.json'),
    ('data/evaluation/word_pairs.json', 'word_pairs.json'),
]

def download_if_missing(local_path: str, asset_name: str):
    p = Path(local_path)
    if p.exists() and p.stat().st_size > 0:
        print(f'✓ Exists: {local_path}')
        return
    p.parent.mkdir(parents=True, exist_ok=True)
    url = f'{RELEASE_BASE}/{asset_name}'
    print(f'Downloading {asset_name} -> {local_path}')
    urllib.request.urlretrieve(url, local_path)
    print(f'✓ Downloaded: {local_path} ({p.stat().st_size:,} bytes)')

for local_path, asset_name in ASSETS:
    download_if_missing(local_path, asset_name)

print('All required data assets are ready ✅')

## قبل التدريب
تأكد أن هذه الملفات موجودة داخل الـrepo أو مرفوعة يدويًا: 
- `results/tokenizers/bpe_16k.model`
- `configs/model/base_30m.yaml`
- `configs/training/default.yaml`
- `data/processed/train_phase1.txt` (لـ Phase1/2)
- `data/morphology/token_root_map.json` (لـ Phase2)

In [ ]:
# 3) Quick check للملفات المطلوبة
from pathlib import Path
required = [
    'notebooks/cloud_train.py',
    'results/tokenizers/bpe_16k.model',
    'configs/model/base_30m.yaml',
    'configs/training/default.yaml',
]
missing = [p for p in required if not Path(p).exists()]
print('Missing:' if missing else 'All required files found ✅')
for p in missing:
    print('-', p)

In [ ]:
# 4) اختيار التجربة (T4 optimized)
EXPERIMENT = 'phase1'  # baseline | phase1 | phase2 | probe
STEPS = 50000
BATCH_SIZE = 64          # T4 target utilization
GRAD_ACCUM_STEPS = 1     # effective batch = 64
PROBE_DIR = 'results/cloud_phase1'  # استخدم مع probe فقط

In [ ]:
# 5) تشغيل التدريب / probing
cmd = [sys.executable, 'notebooks/cloud_train.py', '--experiment', EXPERIMENT]
if EXPERIMENT != 'probe':
    cmd += [
        '--steps', str(STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--grad-accum-steps', str(GRAD_ACCUM_STEPS),
    ]
else:
    cmd += ['--probe-dir', PROBE_DIR]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# 6) عرض المخرجات الأساسية
import glob
from pathlib import Path

for folder in ['results/cloud_baseline', 'results/cloud_phase1', 'results/cloud_phase2']:
    p = Path(folder)
    if p.exists():
        print('\n==', folder, '==')
        for fp in sorted(glob.glob(f'{folder}/*')):
            print('-', fp)

## Tips
- على Kaggle: فعّل GPU من Settings → Accelerator → GPU.
- على Colab: Runtime → Change runtime type → T4 GPU.
- لعمل run سريع للتأكد فقط: اجعل `STEPS = 5`.
- لعمل Probe فقط: `EXPERIMENT='probe'` وغيّر `PROBE_DIR` لمسار التجربة.